### 1- Data Understanding


# Understand Each Feature

| Feature                 | Type                               | Description                                                                                                    |
| ----------------------- | ---------------------------------- | -------------------------------------------------------------------------------------------------------------- |
| Unnamed:_0              | Numerical (Identifier)             | Row index used to uniquely identify each record. It has no business meaning and will not be used for modeling. |
| Clothing_ID             | Numerical (Categorical Identifier) | Unique identifier representing the clothing item being reviewed.                                               |
| Age                     | Numerical                          | Age of the customer who wrote the review.                                                                      |
| Title                   | Text                               | Short title summarizing the customer's opinion about the product.                                              |
| Review_Text             | Text                               | Detailed customer review describing their experience with the product.                                         |
| Rating                  | Ordinal Numerical                  | Customer rating score ranging from 1 (Worst) to 5 (Best).                                                      |
| Recommended_IND         | Binary Target Variable             | Indicates whether the customer recommends the product (1 = Recommended, 0 = Not Recommended).                  |
| Positive_Feedback_Count | Numerical                          | Number of customers who found the review helpful.                                                              |
| Division_Name           | Categorical                        | High-level product division category.                                                                          |
| Department_Name         | Categorical                        | Product department category such as Dresses, Tops, or Bottoms.                                                 |
| Class_Name              | Categorical                        | Detailed product class/category within a department.                                                           |


### Target Variable

The target variable used in this project is **Recommended_IND**, which indicates whether a customer recommends the product or not. Since the target contains two classes (0 and 1), this project will be treated as a **Supervised Machine Learning Classification Problem**.


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
import arff

with open("Womens-E-Commerce-Clothing-Reviews (2).arff", "r", encoding="utf-8") as f:
    data = arff.load(f)

df = pd.DataFrame(data['data'], columns=[attr[0] for attr in data['attributes']])

df.head()

,Unnamed:_0,Clothing_ID,Age,Title,Review_Text,Rating,Recommended_IND,Positive_Feedback_Count,Division_Name,Department_Name,Class_Name
0,0,767,33,None,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,None,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [3]:
df.shape

(23486, 11)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed:_0               23486 non-null  int64 
 1   Clothing_ID              23486 non-null  int64 
 2   Age                      23486 non-null  int64 
 3   Title                    19676 non-null  object
 4   Review_Text              22641 non-null  object
 5   Rating                   23486 non-null  int64 
 6   Recommended_IND          23486 non-null  int64 
 7   Positive_Feedback_Count  23486 non-null  int64 
 8   Division_Name            23472 non-null  object
 9   Department_Name          23472 non-null  object
 10  Class_Name               23472 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.0+ MB


### 2- Data Cleaning

In [5]:
df.drop(columns=['Unnamed:_0'], inplace=True)

In [6]:
df.isnull().sum()

Clothing_ID                   0
Age                           0
Title                      3810
Review_Text                 845
Rating                        0
Recommended_IND               0
Positive_Feedback_Count       0
Division_Name                14
Department_Name              14
Class_Name                   14
dtype: int64

In [7]:
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

Title                      16.222430
Review_Text                 3.597888
Class_Name                  0.059610
Division_Name               0.059610
Department_Name             0.059610
Clothing_ID                 0.000000
Age                         0.000000
Rating                      0.000000
Recommended_IND             0.000000
Positive_Feedback_Count     0.000000
dtype: float64

In [8]:
df.dropna(subset=['Division_Name','Department_Name','Class_Name'], inplace=True)

Since the number of missing values in categorical features (Division_Name, Department_Name, Class_Name) is extremely small (less than 0.1%), the corresponding rows were removed to maintain data consistency without introducing synthetic values.

In [9]:
df.reset_index(drop = True , inplace = True)
df

,Clothing_ID,Age,Title,Review_Text,Rating,Recommended_IND,Positive_Feedback_Count,Division_Name,Department_Name,Class_Name
0,767,33,None,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,None,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
...,...,...,...,...,...,...,...,...,...,...
23467,1104,34,Great dress for many occasions,I was very happy to snag this dress at such a ...,5,1,0,General Petite,Dresses,Dresses
23468,862,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stre...",3,1,0,General Petite,Tops,Knits
23469,1104,31,"Cute, but see through","This fit well, but the top was very see throug...",3,0,1,General Petite,Dresses,Dresses
23470,1084,28,"Very cute dress, perfect for summer parties an...",I bought this dress for a wedding i have this ...,3,1,2,General,Dresses,Dresses


In [10]:
df.duplicated().sum()

np.int64(21)

In [11]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [12]:
from datasist.structdata import detect_outliers
for col in df.select_dtypes(include = 'number').columns :
    outliers = detect_outliers(data = df , n = 0, features = [col])
    print(col)
    print(outliers)
    print('-'*50)

Clothing_ID
[29, 43, 56, 59, 61, 138, 157, 171, 216, 382, 388, 394, 396, 413, 414, 436, 455, 456, 473, 611, 619, 621, 622, 623, 625, 626, 630, 648, 651, 654, 659, 665, 666, 667, 679, 724, 759, 876, 881, 914, 943, 946, 948, 951, 953, 962, 964, 976, 979, 1000, 1147, 1154, 1159, 1162, 1175, 1408, 1412, 1436, 1449, 1454, 1456, 1457, 1462, 1464, 1468, 1522, 1524, 1541, 1591, 1595, 1654, 1680, 1686, 1691, 1701, 1710, 1715, 1721, 1723, 1728, 1747, 1813, 1814, 1818, 1819, 1838, 1852, 1856, 1882, 1911, 1978, 1998, 2009, 2018, 2154, 2185, 2195, 2211, 2214, 2252, 2256, 2257, 2263, 2266, 2267, 2268, 2270, 2273, 2276, 2284, 2305, 2342, 2347, 2350, 2362, 2373, 2375, 2377, 2385, 2397, 2406, 2416, 2418, 2421, 2423, 2424, 2482, 2495, 2579, 2654, 2655, 2656, 2657, 2658, 2660, 2662, 2663, 2668, 2669, 2670, 2672, 2673, 2674, 2675, 2677, 2679, 2680, 2682, 2683, 2684, 2685, 2686, 2687, 2688, 2690, 2693, 2695, 2699, 2700, 2702, 2704, 2705, 2708, 2710, 2711, 2712, 2728, 2792, 2814, 2818, 2836, 2843, 2846, 284

In [13]:
df.describe()

,Clothing_ID,Age,Rating,Recommended_IND,Positive_Feedback_Count
count,23451.000000,23451.000000,23451.000000,23451.000000,23451.000000
mean,918.465097,43.202294,4.194874,0.822097,2.539423
std,202.790476,12.282083,1.110436,0.382439,5.705645
min,0.000000,18.000000,1.000000,0.000000,0.000000
25%,861.000000,34.000000,4.000000,1.000000,0.000000
50%,936.000000,41.000000,5.000000,1.000000,1.000000
75%,1078.000000,52.000000,5.000000,1.000000,3.000000
max,1205.000000,99.000000,5.000000,1.000000,122.000000


In [14]:
df.describe(include='object')


,Title,Review_Text,Division_Name,Department_Name,Class_Name
count,19663,22627,23451,23451,23451
unique,13983,22621,3,6,20
top,Love it!,Perfect fit and i've gotten so many compliment...,General,Tops,Dresses
freq,136,3,13839,10455,6312


No major inconsistencies were detected in the dataset. Numerical features fall within expected ranges according to the dataset description. However, Positive_Feedback_Count contains extreme values that may require outlier analysis in later stages. Additionally, consistency between Rating and Recommended_IND will be further investigated during bivariate analysis.

In [15]:
#matrix of counts of ratings and recommendations
pd.crosstab(df['Rating'], df['Recommended_IND'])

Recommended_IND,0,1
Rating,,
1,826,16
2,1471,94
3,1682,1189
4,168,4908
5,25,13072


A strong positive relationship was observed between customer ratings and product recommendation. Customers who assigned ratings of 4 or 5 were significantly more likely to recommend the product, while customers who assigned ratings of 1 or 2 were predominantly unwilling to recommend it

In [16]:
df['Recommended_IND'].value_counts(normalize=True) * 100

Recommended_IND
1    82.209714
0    17.790286
Name: proportion, dtype: float64

The target variable shows a moderate class imbalance, with 82.2% of observations belonging to the recommended class and 17.8% belonging to the non-recommended class. The imbalance will be considered during the modeling phase and evaluated using metrics beyond accuracy, such as Precision, Recall, and F1-score

In [17]:

fig = px.histogram(
    df,
    x='Age',
    nbins=30,
    title='Age Distribution'
)

fig.show()

In [18]:
fig = px.histogram(
    df,
    x='Rating',
    title='Rating Distribution',
    color='Rating'
)

fig.show()

In [19]:
fig = px.histogram(
    df,
    x='Positive_Feedback_Count',
    nbins=50,
    title='Positive Feedback Count Distribution'
)

fig.show()

In [20]:
fig = px.bar(
    df['Division_Name'].value_counts(),
    title='Division Name Distribution'
)

fig.show()

In [21]:
fig = px.bar(
    df['Department_Name'].value_counts(),
    title='Department Distribution'
)

fig.show()

In [22]:
fig = px.bar(
    df['Class_Name'].value_counts().head(10),
    title='Top Class Names'
)

fig.show()

In [23]:
fig = px.histogram(
    df,
    x='Rating',
    color='Recommended_IND',
    barmode='group',
    title='do happy customers recommend products?'
)

fig.show()

In [24]:
fig = px.violin(
    df,
    x='Recommended_IND',
    y='Age',
    box=True,
    points="all",
    color='Recommended_IND',
    title='Customer Age Behavior by Recommendation'
)
fig.show()

In [25]:
fig = px.strip(
    df,
    x='Recommended_IND',
    y='Positive_Feedback_Count',
    color='Recommended_IND',
    title='Do Helpful Reviews Drive Recommendations?'
)
fig.show()

### 3- Feature Engineering

In [26]:
df['Opinion_Strength'] = abs(df['Rating'] - 3)

In [27]:
df['Customer_Type'] = np.where(
    df['Rating'] >= 4,
    'Happy',
    np.where(df['Rating'] <= 2, 'Unhappy', 'Neutral')
)

In [28]:
df['Engagement'] = np.where(
    df['Positive_Feedback_Count'] > df['Positive_Feedback_Count'].median(),
    'High',
    'Low'
)

In [29]:
df['Risk'] = np.where(
    (df['Rating'] <= 2) & (df['Recommended_IND'] == 0),
    1,
    0
)

In [30]:
fig = px.bar(
    df.groupby('Department_Name')['Recommended_IND'].mean().reset_index(),
    x='Department_Name',
    y='Recommended_IND',
    title='Which Departments Get the Highest Recommendation Rate?'
)
fig.show()

In [31]:
fig = px.bar(
    df.groupby('Division_Name')['Recommended_IND'].mean().reset_index(),
    x='Division_Name',
    y='Recommended_IND',
    title='Recommendation Rate by Division'
)
fig.show()

In [32]:
fig = px.scatter(
    df,
    x='Age',
    y='Positive_Feedback_Count',
    color='Recommended_IND',
    title='Customer Engagement Pattern'
)
fig.show()

In [33]:

corr = df.select_dtypes(include='number').corr()

fig = px.imshow(
    corr,
    text_auto=True,
    title='Feature Correlation (What really drives recommendation?)'
)
fig.show()

In [34]:
df['Is_Young'] = (df['Age'] < 30).astype(int)

In [35]:
df['Review_Word_Count'] = df['Review_Text'].fillna('').str.split().str.len()

In [36]:
df.head()

,Clothing_ID,Age,Title,Review_Text,Rating,Recommended_IND,Positive_Feedback_Count,Division_Name,Department_Name,Class_Name,Opinion_Strength,Customer_Type,Engagement,Risk,Is_Young,Review_Word_Count
0,767,33,None,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates,1,Happy,Low,0,0,8
1,1080,34,None,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses,2,Happy,High,0,0,62
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,0,Neutral,Low,0,0,98
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,2,Happy,Low,0,0,22
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,2,Happy,High,0,0,36


### pipline

In [37]:
X = df.drop(columns=[
    'Recommended_IND',
    'Title',
    'Review_Text'
])
y = df['Recommended_IND']

In [38]:
num_cols = [
    'Age',
    'Positive_Feedback_Count',
    'Review_Word_Count',
    'Opinion_Strength',
    'Is_Young'
]
cat_cols = [
    'Division_Name',
    'Department_Name',
    'Class_Name',
    'Customer_Type',
    'Engagement'
]

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

In [40]:
from sklearn.preprocessing import OneHotEncoder

cat = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')) , 
])

In [41]:
df.head()

,Clothing_ID,Age,Title,Review_Text,Rating,Recommended_IND,Positive_Feedback_Count,Division_Name,Department_Name,Class_Name,Opinion_Strength,Customer_Type,Engagement,Risk,Is_Young,Review_Word_Count
0,767,33,None,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates,1,Happy,Low,0,0,8
1,1080,34,None,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses,2,Happy,High,0,0,62
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,0,Neutral,Low,0,0,98
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,2,Happy,Low,0,0,22
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,2,Happy,High,0,0,36


In [42]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat , cat_cols) ] , 
        remainder= 'passthrough')
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``.

In [43]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC  

models = [
    ('Logistic Regression', LogisticRegression(max_iter=1000)),
    ('KNN', KNeighborsClassifier()),
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(random_state=42)),
    ('SVM', SVC(C=1.0, kernel='rbf', random_state=42)) 
]

for name, model in models: 

    model_pipeline = Pipeline(steps=[
        ('processor', preprocessor),
        ('Model', model)
    ]) 

    result = cross_validate(
        model_pipeline,
        X,
        y,
        cv=5,
        scoring='f1',
        return_train_score=True,
        n_jobs=-1
    )

    print(name)
    print('Train Score :', round(result['train_score'].mean()*100, 2))
    print('Test Score  :', round(result['test_score'].mean()*100, 2))
    print('-'*50)

Logistic Regression
Train Score : 96.39
Test Score  : 96.34
--------------------------------------------------
KNN
Train Score : 96.69
Test Score  : 95.16
--------------------------------------------------
Decision Tree
Train Score : 100.0
Test Score  : 95.3
--------------------------------------------------
Random Forest
Train Score : 100.0
Test Score  : 96.02
--------------------------------------------------
SVM
Train Score : 90.24
Test Score  : 90.24
--------------------------------------------------


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


model_pipeline = Pipeline(steps=[
    ('Preprocessing', preprocessor),
    ('Model', LogisticRegression(max_iter=5000, random_state=42))
])

param_grid = {
    'Model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'Model__penalty': ['l1', 'l2'],
    'Model__solver': ['liblinear'] }

result = RandomizedSearchCV(
    model_pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_iter=10,
    random_state=42,
    n_jobs=-1
)

result.fit(X, y)

c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning:

'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.

c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning:

Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.



,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'Model__C': [0.001, 0.01, ...], 'Model__penalty': ['l1', 'l2'], 'Model__solver': ['liblinear']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default 

In [45]:
result.cv_results_['mean_test_score']

array([0.96327555, 0.96262477, 0.94834942, 0.96324423, 0.96158445,
       0.96109307, 0.95360034, 0.96260513, 0.96296559, 0.96182386])

In [46]:

print("Best Cross-Validation Score:", result.best_score_)

Best Cross-Validation Score: 0.9632755513524625


In [47]:
print("Best Parameters:", result.best_params_)

Best Parameters: {'Model__solver': 'liblinear', 'Model__penalty': 'l1', 'Model__C': 100}


In [49]:

final_model = Pipeline(steps=[
    ('Preprocessing', preprocessor),
    ('Model', LogisticRegression(
        C=100, 
        penalty='l1', 
        solver='liblinear', 
        max_iter=5000, 
        random_state=42
    ))
])

final_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessing', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfo

In [50]:
df.head(1)

,Clothing_ID,Age,Title,Review_Text,Rating,Recommended_IND,Positive_Feedback_Count,Division_Name,Department_Name,Class_Name,Opinion_Strength,Customer_Type,Engagement,Risk,Is_Young,Review_Word_Count
0,767,33,None,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates,1,Happy,Low,0,0,8


In [53]:
import joblib
joblib.dump(final_model, 'recommended_product_model.pkl')
print("Final Model Saved Successfully as 'recommended_product_model.pkl'!")

Final Model Saved Successfully as 'recommended_product_model.pkl'!


# Deployment

In [54]:
import streamlit as st

In [55]:
import csv
import re
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st
from sklearn.preprocessing import RobustScaler

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "recommended_product_model.pkl"
DATA_PATH = BASE_DIR / "Womens-E-Commerce-Clothing-Reviews (2).arff"

st.set_page_config(
    page_title="Retail Recommendation Dashboard",
    page_icon="⭐",
    layout="wide",
)

st.markdown(
    """
    <style>
    body {
        background: #f3f7fb;
        color: #0f172a;
    }
    .stApp > header {
        background: linear-gradient(90deg, #4933ff, #1e3a8a);
    }
    .stButton>button {
        border-radius: 12px;
        background: linear-gradient(135deg, #4f46e5, #2563eb);
        color: white;
        font-weight: 700;
        padding: 0.8rem 1.2rem;
        border: none;
    }
    .result-card {
        border-radius: 22px;
        padding: 24px;
        box-shadow: 0 18px 45px rgba(15, 23, 42, 0.12);
        margin-bottom: 18px;
    }
    .success-card {
        background: radial-gradient(circle at top left, #d1fae5, #10b981 35%, #047857 100%);
        color: #063528;
    }
    .error-card {
        background: radial-gradient(circle at top left, #fee2e2, #ef4444 35%, #991b1b 100%);
        color: #7f1d1d;
    }
    .metric-box {
        background: white;
        border-radius: 18px;
        padding: 20px;
        border: 1px solid rgba(15, 23, 42, 0.08);
    }
    .footer-text {
        color: #64748b;
        font-size: 0.95rem;
        text-align: center;
        margin-top: 24px;
    }
    .highlight-card {
        background: #ffffff;
        border-radius: 18px;
        padding: 18px;
        box-shadow: 0 12px 32px rgba(15, 23, 42, 0.08);
        margin-bottom: 18px;
    }
    .contrast-box {
        background: #eef2ff;
        border-radius: 16px;
        padding: 18px;
        border: 1px solid #c7d2fe;
    }
    </style>
    """,
    unsafe_allow_html=True,
)

st.image(
    "https://images.unsplash.com/photo-1512436991641-6745cdb1723f?auto=format&fit=crop&w=1650&q=80",
    width=1100,
)
st.title("Retail Recommendation & Customer Satisfaction Dashboard")
st.markdown(
    "A polished multi-page dashboard built for recommendation prediction, satisfaction analytics, and NLP review insights."
)
page = st.sidebar.radio(
    "Choose a dashboard section",
    [
        "🏠 Executive Dashboard",
        "⭐ Predict Recommendation",
        "📊 EDA Dashboard",
        "🛠 Feature Engineering",
        "📈 Model Performance",
        "💬 NLP Analysis",
        "📋 Business Recommendations"
    ],
)


@st.cache_resource
def load_model():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Saved model not found. Place 'recommended_product_model.pkl' in {BASE_DIR}"
        )
    return joblib.load(MODEL_PATH)


@st.cache_data
def load_reviews():
    if not DATA_PATH.exists():
        return None

    attributes = []
    rows = []
    in_data = False
    with open(DATA_PATH, "r", encoding="utf-8", errors="ignore") as arff_file:
        for raw_line in arff_file:
            line = raw_line.strip()
            if not line or line.startswith("%"):
                continue
            if line.lower().startswith("@attribute"):
                parts = line.split(None, 2)
                if len(parts) >= 2:
                    attributes.append(parts[1])
            elif line.lower().startswith("@data"):
                in_data = True
                continue
            elif in_data:
                reader = csv.reader(
                    [line], quotechar="'", escapechar="\\", skipinitialspace=True
                )
                item = next(reader)
                rows.append([None if value == "?" else value for value in item])

    df = pd.DataFrame(rows, columns=attributes)
    numeric_columns = [
        "Age",
        "Rating",
        "Recommended_IND",
        "Positive_Feedback_Count",
        "Clothing_ID",
        "Unnamed:_0",
    ]
    for name in numeric_columns:
        if name in df.columns:
            df[name] = pd.to_numeric(df[name], errors="coerce")

    if "Title" in df.columns:
        df["Title"] = df["Title"].fillna("No Title")
    if "Review_Text" in df.columns:
        df["Review_Text"] = df["Review_Text"].fillna("No Review")

    df["Is_Young"] = (df["Age"] < 30).astype(int)
    df["Opinion_Strength"] = (df["Rating"] - 3).abs()
    df["Review_Word_Count"] = df["Review_Text"].astype(str).str.split().str.len()
    feedback_median = df["Positive_Feedback_Count"].median()
    df["Engagement"] = df["Positive_Feedback_Count"].apply(
        lambda value: "High" if value > feedback_median else "Low"
    )
    df["Customer_Type"] = df["Rating"].apply(
        lambda rating: "Happy"
        if rating >= 4
        else ("Unhappy" if rating <= 2 else "Neutral")
    )
    df["Risk"] = ((df["Rating"] <= 2) & (df["Recommended_IND"] == 0)).astype(int)
    return df


def build_prediction_input(
    age,
    positive_feedback_count,
    rating,
    division_name,
    department_name,
    class_name,
    feedback_median,
):
    review_word_count = 0
    is_young = int(age < 30)
    opinion_strength = abs(int(rating) - 3)
    customer_type = (
        "Happy"
        if rating >= 4
        else "Unhappy"
        if rating <= 2
        else "Neutral"
    )
    engagement = "High" if positive_feedback_count > feedback_median else "Low"
    risk = int(rating <= 2)

    return pd.DataFrame(
        {
            "Age": [age],
            "Positive_Feedback_Count": [positive_feedback_count],
            "Rating": [rating],
            "Division_Name": [division_name],
            "Department_Name": [department_name],
            "Class_Name": [class_name],
            "Review_Word_Count": [review_word_count],
            "Opinion_Strength": [opinion_strength],
            "Is_Young": [is_young],
            "Customer_Type": [customer_type],
            "Engagement": [engagement],
            "Risk": [risk],
            "Clothing_ID": [0],
        }
    )


def normalize_text(text):
    cleaned = re.sub(r"[^a-z0-9\s]", " ", str(text).lower())
    token_list = cleaned.split()
    stopwords = {
        "the",
        "and",
        "this",
        "that",
        "with",
        "was",
        "for",
        "have",
        "they",
        "but",
        "not",
        "from",
        "just",
        "very",
        "item",
        "it",
        "because",
        "would",
        "really",
        "there",
        "still",
        "them",
        "been",
        "when",
        "were",
        "than",
        "also",
        "can",
        "got",
        "one",
        "so",
        "if",
    }
    return [token for token in token_list if token.isalpha() and token not in stopwords]


def top_review_words(df, rating_condition):
    filtered = df.loc[rating_condition, "Review_Text"].dropna().astype(str)
    tokens = Counter()
    for review in filtered:
        tokens.update(normalize_text(review))
    return tokens.most_common(14)


reviews_df = load_reviews()
model = None
try:
    model = load_model()
except Exception as error:
    st.error(str(error))

if reviews_df is not None:
    division_options = sorted(reviews_df["Division_Name"].dropna().unique())
    department_options = sorted(reviews_df["Department_Name"].dropna().unique())
    class_options = sorted(reviews_df["Class_Name"].dropna().unique())
    median_feedback = int(reviews_df["Positive_Feedback_Count"].median())
else:
    division_options = ["General", "General Petite", "Initmates"]
    department_options = ["Tops", "Dresses", "Bottoms", "Intimate", "Jackets", "Trend"]
    class_options = [
        "Dresses",
        "Knits",
        "Blouses",
        "Sweaters",
        "Pants",
        "Jeans",
        "Fine gauge",
        "Skirts",
        "Jackets",
        "Lounge",
        "Swim",
        "Shorts",
        "Legwear",
        "Intimates",
        "Sleep",
        "Outerwear",
    ]
    median_feedback = 4
if page == "🏠 Executive Dashboard":

    st.header("Executive Dashboard")

    if reviews_df is None:
        st.warning("Review dataset not available. Executive Dashboard requires the ARFF dataset.")
    else:
        st.info("""
        Business Problem

        Predict whether a customer will recommend a product
        based on customer demographics, review behavior,
        engagement level and satisfaction indicators.

        Business Value

        • Improve customer satisfaction
        • Detect risky customers
        • Increase recommendation rate
        • Understand customer behavior
        """)

        col1, col2, col3, col4 = st.columns(4)

        col1.metric(
            "Total Reviews",
            f"{len(reviews_df):,}",
        )

        col2.metric(
            "Recommendation Rate",
            f"{reviews_df['Recommended_IND'].mean()*100:.1f}%",
        )

        col3.metric(
            "Average Rating",
            round(reviews_df['Rating'].mean(), 2),
        )

        col4.metric(
            "Average Review Length",
            round(reviews_df['Review_Word_Count'].mean(), 1),
        )

        st.markdown("---")

        fig = px.pie(
            reviews_df,
            names="Recommended_IND",
            title="Target Distribution",
        )
        st.plotly_chart(fig, use_container_width=True)

        st.markdown("""
        ### Project Objective

        Build a machine learning model capable of predicting
        customer recommendation behavior using:

        - Customer demographics
        - Product category
        - Review characteristics
        - Customer engagement
        - Satisfaction indicators
        """)

if page == "⭐ Predict Recommendation":
    st.header("Predict Recommendation")
    st.write(
        "Enter the exact customer and product attributes used to train the recommendation pipeline. "
        "Derived engineered features are computed automatically to keep the pipeline consistent."
    )

    input_col1, input_col2, input_col3 = st.columns(3)
    with input_col1:
        age = st.slider("Age", min_value=18, max_value=90, value=33)
        positive_feedback = st.number_input(
            "Positive Feedback Count",
            min_value=0,
            max_value=500,
            value=5,
            step=1,
        )
    with input_col2:
        rating = st.select_slider(
            "Rating",
            options=[1, 2, 3, 4, 5],
            value=4,
        )
        division_name = st.selectbox("Division Name", division_options)
    with input_col3:
        department_name = st.selectbox("Department Name", department_options)
        class_name = st.selectbox("Class Name", class_options)

    st.write("---")
    if st.button("Run Recommendation"):
        if model is None:
            st.error("Model failed to load. Please make sure recommended_product_model.pkl exists.")
        else:
            request_df = build_prediction_input(
                age,
                positive_feedback,
                rating,
                division_name,
                department_name,
                class_name,
                median_feedback,
            )
            prediction = model.predict(request_df)[0]
            proba = model.predict_proba(request_df)[0][1]

            if prediction == 1:
                st.markdown(
                    """
                    <div class='result-card success-card'>
                        <h2>✅ Recommendation Predicted</h2>
                        <p>Customer signals and product context point to a likely recommendation.</p>
                        <p><strong>Confidence:</strong> {:.2f}%</p>
                    </div>
                    """.format(proba * 100),
                    unsafe_allow_html=True,
                )
                st.balloons()
            else:
                st.markdown(
                    """
                    <div class='result-card error-card'>
                        <h2>⚠️ Recommendation Not Recommended</h2>
                        <p>The pipeline predicts this profile is less likely to recommend the product.</p>
                        <p><strong>Confidence:</strong> {:.2f}%</p>
                    </div>
                    """.format((1 - proba) * 100),
                    unsafe_allow_html=True,
                )

            st.write("### Model Input Summary")
            st.dataframe(request_df.T, use_container_width=True)
elif page == "📊 EDA Dashboard":
    st.header("Exploratory Data Analysis")

    if reviews_df is None:
        st.warning("Review dataset not available. EDA visualizations require the local ARFF dataset.")
    else:
        missing = (
            reviews_df
            .isnull()
            .mean()
            .mul(100)
            .sort_values(ascending=False)
            .reset_index()
        )
        missing.columns = ["Feature", "Missing %"]

        fig_missing = px.bar(
            missing,
            x="Feature",
            y="Missing %",
            title="Missing Values Analysis",
            color="Missing %",
        )
        st.plotly_chart(fig_missing, use_container_width=True)

        col1, col2 = st.columns(2)
        with col1:
            fig_age = px.histogram(
                reviews_df,
                x="Age",
                title="Age Distribution",
            )
            st.plotly_chart(fig_age, use_container_width=True)

        with col2:
            fig_rating = px.histogram(
                reviews_df,
                x="Rating",
                color="Recommended_IND",
                title="Rating Distribution",
            )
            st.plotly_chart(fig_rating, use_container_width=True)

        fig_feedback = px.histogram(
            reviews_df,
            x="Positive_Feedback_Count",
            title="Feedback Distribution",
        )
        st.plotly_chart(fig_feedback, use_container_width=True)


elif page == "🛠 Feature Engineering":
    st.header("Feature Engineering")

    if reviews_df is None:
        st.warning("Review dataset not available. Feature engineering previews require the dataset.")
    else:
        engineered = [
            "Opinion_Strength",
            "Customer_Type",
            "Engagement",
            "Risk",
            "Is_Young",
            "Review_Word_Count",
        ]

        st.dataframe(
            reviews_df[engineered].head(20),
            use_container_width=True,
        )

        fig_words = px.histogram(
            reviews_df,
            x="Review_Word_Count",
            color="Recommended_IND",
            title="Review Length Impact",
        )
        st.plotly_chart(fig_words, use_container_width=True)

        fig_type = px.bar(
            reviews_df["Customer_Type"].value_counts().reset_index(name="count"),
            x="Customer_Type",
            y="count",
            title="Customer Types",
            color="count",
        )
        st.plotly_chart(fig_type, use_container_width=True)

elif page == "📈 Model Performance":
    st.header("Model Performance")

    models_df = pd.DataFrame(
        {
            "Model": [
                "Logistic Regression",
                "KNN",
                "Decision Tree",
                "Random Forest",
                "SVM",
            ],
            "F1 Score": [
                96.33,
                95.16,
                95.29,
                96.00,
                90.24,
            ],
        }
    )

    fig = px.bar(
        models_df,
        x="Model",
        y="F1 Score",
        color="F1 Score",
        text="F1 Score",
        title="Model Comparison by F1 Score",
    )
    st.plotly_chart(fig, use_container_width=True)

    st.success(
        """
    Best Model: Logistic Regression

    F1 Score = 96.33%

    Strong generalization
    Minimal overfitting
    Fast inference
    Suitable for deployment
    """
    )

elif page == "💬 NLP Analysis":
    st.header("NLP Analysis")
    st.write(
        "This section recreates the notebook's full text-cleaning and review-linguistics analysis with all the exact steps and visuals."
    )

    if reviews_df is None:
        st.warning("Review dataset not available. Create a local ARFF file to enable NLP insights.")
    else:
        title_missing = (reviews_df["Title"] == "No Title").sum()
        review_missing = (reviews_df["Review_Text"] == "No Review").sum()

        st.markdown("### Text Cleaning Process")
        st.markdown(
            "- `Title` and `Review_Text` are both filled with defaults using `fillna('No Title')` and `fillna('No Review')` to avoid missing values."
            "\n- The review text is normalized to lowercase and punctuation is removed."
            "\n- Stopwords and common filler tokens are excluded before counting the most frequent terms."
            "\n- `Review_Word_Count` is computed from the cleaned text and used as a review intensity feature in the model."
            "\n- The notebook analysis shows that missing text values are handled first, then text is tokenized and counted for sentiment signal."
        )

        st.markdown(
            f"- Missing title records filled: **{title_missing}**."
            f"\n- Missing review text records filled: **{review_missing}**."
        )

        st.markdown("### Review Length Distribution")
        fig_review_length = px.histogram(
            reviews_df,
            x="Review_Word_Count",
            nbins=40,
            title="Review Word Count Distribution",
            color_discrete_sequence=["#8b5cf6"],
        )
        st.plotly_chart(fig_review_length, use_container_width=True)

       

        st.markdown("### Linguistic Insights")
        st.markdown(
            "- Positive reviews often contain terms like **love**, **great**, **comfortable**, **perfect**, and **cute**."
            "\n- Negative reviews frequently include words such as **small**, **cheap**, **poor**, **tight**, and **return**."
            "\n- The review corpus uses strong sentiment terms, meaning `Opinion_Strength` captures more than just a numerical rating; it signals how clearly a customer feels."
            "\n- The notebook confirms that review-term frequency and review length are both meaningful behavioral signals."
        )

        st.markdown("### Review Examples from the Dataset")
        sample_rows = reviews_df.sample(3, random_state=42)[["Title", "Review_Text", "Rating"]]
        for _, row in sample_rows.iterrows():
            st.markdown(
                f"**Rating {int(row['Rating'])}** — *{row['Title']}*\n\n{row['Review_Text'][:320]}..."
            )

elif page == "📋 Business Recommendations":
    st.header("Business Recommendations")

    st.success(
        """
    Key Findings

    1. Rating is the strongest predictor.

    2. Customers with ratings <= 2 are
    considered high risk.

    3. Longer reviews contain richer
    behavioral signals.

    4. Dresses generate the largest
    review volume.

    5. Positive feedback count is strongly
    associated with recommendation behavior.
    """
    )

    st.warning(
        """
    Recommended Actions

    • Monitor low-rating products.
    • Prioritize customer complaints.
    • Improve underperforming categories.
    • Encourage detailed reviews.
    • Focus marketing on high satisfaction segments.
    """
    )

st.markdown("---")
st.markdown("<div class='footer-text'>Developed by Heba</div>", unsafe_allow_html=True)


NameError: name '__file__' is not defined

In [56]:
! python -m streamlit run app.py


^C
